# Hands-on 1 — IceCube HESE: physics weights and detector response

**PhD school · analysis techniques in neutrino astronomy**  
**INSTRUCTOR SOLUTIONS — aligned to the revised student notebook**  
Based on the lecture *Analysis Techniques in Neutrino Astronomy*, especially slides **21–41 (MC weights), 42–53 (selection), 56–61 (effective area), 63–66 (energy response), 70–81 (direction and PSF)**.

**Learning goals:** (i) contrast MC and observed events; (ii) turn a physics flux into expected event counts, including exposure; (iii) compare atmospheric and astrophysical components; (iv) derive effective area, energy/zenith migration, and a conditional topology response from selected MC.

**Official sources:** [IceCube 7.5-year HESE release](https://icecube.wisc.edu/data-releases/2021/12/hese-7-5-year-data/) · [Variable dictionary](https://github.com/icecube/HESE-7-year-data-release) · [paper](https://doi.org/10.1103/PhysRevD.104.022002).

> The archive has **102 observed events** and MC entries used for **the selected HESE sample**. Reconstructed quantities are available in data and MC, but truth and flux weights only in MC. The release does not include the azimuths or a full pre-selection/generated MC record.

## A. Set up the official data release

Run the following cells. The notebook first searches for locally extracted `HESE_*.json` files, then downloads the 75-MB official ZIP if needed. If internet is unavailable, download the ZIP from the [official release](https://icecube.wisc.edu/data-releases/2021/12/hese-7-5-year-data/) and put it beside this notebook.

**Units/meanings to inspect in the README:**

| Field | Meaning / units |
|---|---|
| `primaryEnergy`, `primaryZenith`, `primaryType`, `interactionType` | MC truth: GeV, rad, PDG code, interaction category |
| `recoDepositedEnergy`, `recoZenith`, `recoMorphology`, `recoLength` | Data and MC reconstructed: GeV, rad, 0=cascade/1=track/2=double cascade, m |
| `weightOverFluxOverLivetime` | neutrino normalization factor; **GeV sr cm²** (zero for muons) |
| `pionFlux`, `kaonFlux`, `promptFlux` | atmospheric differential flux per MC event; **GeV⁻¹ s⁻¹ sr⁻¹ cm⁻²** |
| `conventionalSelfVetoCorrection`, `promptSelfVetoCorrection` | dimensionless veto passing fractions |
| `muonWeightOverLivetime` | atmospheric muon expected rate contribution (s⁻¹), zero for neutrinos |

**Caution about flux conventions:** `PHI0` below is an **illustrative differential flux for each particle species represented by a positive neutrino MC entry**, not the published all-flavor HESE best-fit normalization.

### Particle identification: PDG codes

In the Monte Carlo sample, the variable `primaryType` identifies the simulated primary particle using the **Particle Data Group (PDG) code**.

Official documentation:  
https://pdg.lbl.gov/2025/reviews/rpp2025-rev-monte-carlo-numbering.pdf


| Particle | PDG code |
|---|---:|
| Electron neutrino ($\nu_e$) | `12` |
| Electron antineutrino ($\bar{\nu}_e$) | `-12` |
| Muon neutrino ($\nu_\mu$) | `14` |
| Muon antineutrino ($\bar{\nu}_\mu$) | `-14` |
| Tau neutrino ($\nu_\tau$) | `16` |
| Tau antineutrino ($\bar{\nu}_\tau$) | `-16` |
| Muon ($\mu^-$) | `13` |
| Antimuon ($\mu^+$) | `-13` |

**Note:** Neutrinos have positive PDG codes, while antineutrinos have negative codes.

To select a neutrino flavour including both neutrinos and antineutrinos, use the absolute value of `primaryType`.

For example, to select all muon neutrinos and antineutrinos:

```python
mask = np.abs(df_mc["primaryType"]) == 14
```

### Interaction categories in the HESE MC

The `interactionType` variable distinguishes the simulated neutrino interaction channel:

| `interactionType` | Meaning |
|---:|---|
| `0` | Simulated atmospheric muon (not a neutrino interaction) |
| `1` | Charged-current neutrino interaction (CC) |
| `2` | Neutral-current neutrino interaction (NC) |
| `3` | Glashow-resonance interaction (GR) |

Reference: [IceCube HESE release — variable descriptions](https://github.com/icecube/HESE-7-year-data-release).


In [ ]:
import json
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

plt.rcParams.update({'figure.figsize':(8, 4.8), 'axes.grid':True, 'grid.alpha':0.22})
FILES = ('HESE_data.json', 'HESE_mc_truth.json', 'HESE_mc_observable.json', 'HESE_mc_flux.json')
DATA_DIR = Path('HESE_7p5_data')
ARCHIVE_PATH = Path('20211217_HESE-7-5-year-data.zip')
ARCHIVE_URL = 'https://icecube.wisc.edu/data-releases/20211217_HESE-7-5-year-data.zip'
LIVETIME_S = 227708167.68  # seconds: historical 7.5-year HESE analysis; see release code
E0_GEV = 1e5  # 100 TeV
PHI0 = 1e-18  # GeV^-1 cm^-2 s^-1 sr^-1, illustrative PER-PARTICLE-SPECIES normalization

def prepare_release(target=DATA_DIR):
    """Prefer local files; otherwise download/extract the four official JSON files."""
    target.mkdir(exist_ok=True)
    if all((target / f).is_file() for f in FILES):
        print('Using existing HESE files:', target.resolve())
        return target
    for base in (Path('resources/data'),
                 Path('HESE-7-year-data-release/resources/data'),
                 Path('HESE-7-year-data-release/HESE-7-year-data-release/resources/data')):
        if all((base / f).is_file() for f in FILES):
            print('Using existing extracted release:', base.resolve())
            return base
    if not ARCHIVE_PATH.is_file():
        try:
            print('Downloading the 75-MB official IceCube ZIP (once).')
            urlretrieve(ARCHIVE_URL, ARCHIVE_PATH)
        except Exception as exc:
            raise RuntimeError('Automatic download unavailable. Download the ZIP manually from '
                               'https://icecube.wisc.edu/data-releases/2021/12/hese-7-5-year-data/ '
                               'and save it beside this notebook as '+str(ARCHIVE_PATH)) from exc
    with ZipFile(ARCHIVE_PATH) as z:
        for f in FILES:
            matches = [name for name in z.namelist() if name.endswith('/resources/data/'+f)
                       or name == 'resources/data/'+f]
            if len(matches) != 1:
                raise ValueError(f'Expected one {f} in the archive, found {len(matches)}')
            with z.open(matches[0]) as source, (target/f).open('wb') as dest:
                shutil.copyfileobj(source, dest)
    print('Extracted files to', target.resolve())
    return target

def load_json(path):
    with path.open(encoding='utf-8') as f:
        return {k: np.asarray(v) for k, v in json.load(f).items()}

data_dir = prepare_release()
df_data = pd.DataFrame(load_json(data_dir / 'HESE_data.json'))
mc_columns = {}
for filename in FILES[1:]:
    record = load_json(data_dir/filename)
    overlap = set(mc_columns).intersection(record)
    if overlap:
        raise ValueError(f'Overlapping MC column names in {filename}: {overlap}')
    mc_columns.update(record)
df_mc = pd.DataFrame(mc_columns)
assert len(df_data) == 102, 'Unexpected number of observed HESE events: check the release.'
assert len(df_mc) > 0
print(f'{len(df_data)} observed events; {len(df_mc):,} selected MC entries')
print('Data columns:', df_data.columns.tolist())
print('MC columns:', df_mc.columns.tolist())

## B. From generation weight to event rate

The released `weightOverFluxOverLivetime` contains the relevant MC normalization. **Data** have one count per event; **MC** have model-dependent counts after multiplying by livetime. The notebook uses `T = 227708167.68 s` from the historical analysis.

Atmospheric muons have a separately supplied `muonWeightOverLivetime`; multiplying them by a neutrino flux would be incorrect.

### Exercise 1 — From flux to expected events

In the previous lecture, we discussed how Monte Carlo events can be reweighted to represent different physical flux hypotheses.

**Your task:**

1. Implement the function `astro_counts()` to calculate the expected contribution of each MC event for a given astrophysical flux.
2. Inspect the units of `weightOverFluxOverLivetime`. What are the units of the resulting weights after multiplying by the flux?
3. How do you obtain the expected number of detected events during the observation period?
4. Calculate the expected number of events for spectral indices $\gamma = 2.0,\ 2.7,\ 3.8$.
5. Which spectral index predicts the largest number of events? Does this depend on the reference energy and flux normalization?

**Hint:** Recall the relationship between generation weights, physical flux, and observation time discussed in the lecture.

In [ ]:
E_true = df_mc['primaryEnergy'].to_numpy(dtype=float)
w0 = df_mc['weightOverFluxOverLivetime'].to_numpy(dtype=float)
T = LIVETIME_S
is_neutrino = (w0 > 0) & np.isfinite(E_true) & (E_true > 0)

def astro_flux(energy_gev, gamma, phi0=None):
    """Differential flux PER SIMULATED PARTICLE SPECIES (not all-flavor sum)."""
    if phi0 is None:
        phi0 = PHI0  # read the current default if students change PHI0 later
    energy_gev = np.asarray(energy_gev, dtype=float)
    with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
        return np.where(energy_gev > 0, phi0*(energy_gev/E0_GEV)**(-gamma), 0.)

def astro_counts(gamma, phi0=None):
    """
    Compute the expected number of astrophysical neutrino
    events for a given spectral index.
    """
    # Rate per MC event [s^-1] = w0 [GeV sr cm^2] x flux [GeV^-1 s^-1 sr^-1 cm^-2].
    # Multiplication by livetime T [s] gives expected event counts (dimensionless).
    flux = astro_flux(E_true, gamma, phi0=phi0)
    counts = np.nan_to_num(T * w0 * flux, nan=0., posinf=0., neginf=0.)
    return np.where(is_neutrino, counts, 0.)

def background_counts(conv_scale=1., prompt_scale=1., muon_scale=1., self_veto=True):
    """Event counts from nominal atm neutrino fluxes and muon MC."""
    d = df_mc
    conv_flux = (d['pionFlux'].to_numpy(float) + d['kaonFlux'].to_numpy(float))
    prompt_flux = d['promptFlux'].to_numpy(float)
    if self_veto:
        conv_flux = conv_flux * d['conventionalSelfVetoCorrection'].to_numpy(float)
        prompt_flux = prompt_flux * d['promptSelfVetoCorrection'].to_numpy(float)
    conv = np.where(is_neutrino,
                    np.nan_to_num(T * w0 * conv_flux * conv_scale, nan=0., posinf=0., neginf=0.),
                    0.)
    prompt = np.where(is_neutrino,
                      np.nan_to_num(T * w0 * prompt_flux * prompt_scale, nan=0., posinf=0., neginf=0.),
                      0.)
    muon = np.nan_to_num(T * d['muonWeightOverLivetime'].to_numpy(float) * muon_scale)
    return {'Conventional atm. ν': conv, 'Prompt atm. ν': prompt, 'Atm. μ': muon}

def plot_data_mc(variable, bins, components, x_label, logx=False, logy=False):
    """Data have unit weight; prediction is weighted by EXPECTED EVENTS."""
    fig, ax = plt.subplots()
    x = df_mc[variable].to_numpy(dtype=float)
    valid = np.isfinite(x) & (x > 0 if logx else True)
    for label, w in components.items():
        ww = np.asarray(w, dtype=float)
        mask = valid & np.isfinite(ww)
        ax.hist(x[mask], bins=bins, weights=ww[mask], histtype='step', lw=1.9, label=label)
    y = df_data[variable].to_numpy(dtype=float)
    y = y[np.isfinite(y) & (y > 0 if logx else True)]
    n, edges = np.histogram(y, bins=bins)
    centers = np.sqrt(edges[:-1]*edges[1:]) if logx else (edges[:-1]+edges[1:])/2
    ax.errorbar(centers, n, yerr=np.sqrt(n), fmt='ko', ms=4, label=f'Observed ({len(df_data)} events)')
    ax.set(xlabel=x_label, ylabel='Events per bin')
    if logx: ax.set_xscale('log')
    if logy: ax.set_yscale('log')
    ax.legend(fontsize=8)
    plt.show()

def weighted_quantile(values, weights, quantiles):
    values = np.asarray(values, float)
    weights = np.asarray(weights, float)
    good = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not np.any(good): return np.full(len(quantiles), np.nan)
    order = np.argsort(values[good])
    v, w = values[good][order], weights[good][order]
    return np.interp(np.asarray(quantiles), (np.cumsum(w) - 0.5*w)/w.sum(), v)


In [ ]:
# Exercise 1 — verification of the reweighting calculation
print('Astrophysical event rate (1/s):', np.sum(astro_counts(2.7)) / T)
print('Astrophysical event expectation during the livetime:', np.sum(astro_counts(2.7)))
for gamma in [2.0, 2.7, 3.8]:
    print(f'gamma={gamma:g}: expected N = {np.sum(astro_counts(gamma)):.4g}')
assert np.all(astro_counts(2.7)[~is_neutrino] == 0)


**Instructor notes — Exercise 1.** By the official HESE variable convention,
 $[w_0]=\mathrm{GeV\,sr\,cm^2}$, $[\Phi]=\mathrm{GeV^{-1}\,s^{-1}\,sr^{-1}\,cm^{-2}}$,
 so $[w_0\Phi]=\mathrm{s^{-1}}$ and $[T w_0\Phi]=1$ (an expected event count).
 The expected number is $\sum_i T w_{0,i}\Phi(E_i)$ over selected neutrino MC entries,
 and atmospheric muons instead use `muonWeightOverLivetime`. This is an **illustrative
 species-by-species power law**, not a fitted HESE flux. The printed totals are calculated
 from the loaded release rather than hard-coded.

### Exercise 2 — explore the sample and compare data with MC

1. Print the observed/MC counts and available fields. Which fields are **truth** and which are **reconstructed**?
2. Plot `recoDepositedEnergy`, `recoZenith` (or `cos(recoZenith)`), and `recoMorphology` for **observed data and MC**.
3. Overlay weighted MC components for an astrophysical power law at $\gamma=2.7$ and nominal atmospheric conventional, prompt, and muon backgrounds. Use identical bins for data/MC. What changes if the astro index is 2.0 or 3.8? Why must you not compare raw MC counts directly to data?
4. Add a **Total MC prediction** (the sum of the astrophysical, conventional, prompt, and atmospheric-muon component weights) to each data/MC plot. Compare the **total** with observed data, not just the individual components. Once you have created `components` with its physical components, you can use:

   ```python
   components["Total MC"] = np.sum(list(components.values()), axis=0)
   ```

   Add the total **only once**, after the individual components have been defined.
5. Optional: repeat for tracks vs cascades only; investigate missing `recoLength` values.

*Plotting note:* the helper uses `sqrt(n)` for observed-event error bars. This is a convenient first approximation; at very low counts, exact Poisson intervals are more appropriate.


In [ ]:
# Exercise 2 — truth/reconstruction and observed-vs-predicted distributions
print('Observed events:', len(df_data), 'Selected MC entries:', len(df_mc))
print('Observed fields:', df_data.columns.tolist())
print('MC fields:', df_mc.columns.tolist())
print('\nObserved reconstructed quantities:')
print(df_data[['recoDepositedEnergy', 'recoZenith', 'recoMorphology']].describe())

# Use physical expected-event weights. Include each physical component exactly once.
components = {'Astrophysical ν, γ=2.7': astro_counts(2.7), **background_counts()}
components['Total MC'] = np.sum(list(components.values()), axis=0)
print('\nComponent event expectations in the release livetime:')
for name, weights in components.items():
    print(f'{name:25s}: {weights.sum():.4g}')

plot_data_mc('recoDepositedEnergy', np.geomspace(5e3, 2e7, 19), components,
             'Reconstructed deposited energy [GeV]', logx=True)
plot_data_mc('recoZenith', np.linspace(0, np.pi, 17), components,
             'Reconstructed zenith [rad]')
plot_data_mc('recoMorphology', np.array([-0.5, 0.5, 1.5, 2.5]), components,
             'Reconstructed morphology (0=cascade, 1=track, 2=double cascade)')

# Illustrate how a different spectral index changes the *detected* energy projection.
for gamma in [2.0, 3.8]:
    components_gamma = {f'Astrophysical ν, γ={gamma:g}': astro_counts(gamma),
                        **background_counts()}
    components_gamma['Total MC'] = np.sum(list(components_gamma.values()), axis=0)
    plot_data_mc('recoDepositedEnergy', np.geomspace(5e3, 2e7, 19), components_gamma,
                 f'Reconstructed deposited energy [GeV] (γ={gamma:g})', logx=True)


**Instructor notes — Exercise 2.** In data, the `reco*` columns describe reconstructed
 event properties; MC also has `primary*` truth fields and flux/normalization weights.
 Comparing data with MC requires **the sum of the expected astrophysical and atmospheric
 contributions**, not the raw number of simulated entries. Do not add `Total MC` into the
 component dictionary a second time: that would double-count. The observed error bars use
 $\sqrt{n}$ as a teaching approximation, not a low-count Poisson interval. The results
 change with the chosen flux and normalization; this is not an inference or best fit.

### Exercise 3 — what do the weights actually represent?

1. Using the units in the release README, show why `w0 × flux` is in $\mathrm{s^{-1}}$ and `T × w0 × flux` is dimensionless.
2. Compare raw MC `primaryEnergy` counts to weighted expected-event distributions for $\gamma=2,2.7,3.8$. Why does a raw MC histogram **not** represent the physical event rate?
3. Calculate `sum(astro_counts(γ))` for $\gamma=2,2.7,3.8$; split each sum into **below** and **above** $E_0=100\,\mathrm{TeV}$. Which index produces more events, and does a steeper spectrum always mean fewer?
4. Explain how changing the flux pivot, normalization, and true-energy acceptance influences the answer. Keep the same $\Phi(E_0)$ while changing $\gamma$.

In [ ]:
# Exercise 3 — raw selected MC entries versus physical counts
indices = [2.0, 2.7, 3.8]
rows = []
for gamma in indices:
    weights = astro_counts(gamma)
    rows.append({'gamma': gamma,
                 'N expected': weights.sum(),
                 'N (Etrue < 100 TeV)': weights[is_neutrino & (E_true < E0_GEV)].sum(),
                 'N (Etrue >= 100 TeV)': weights[is_neutrino & (E_true >= E0_GEV)].sum()})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f'{x:.4g}'))

true_bins = np.geomspace(1e2, 1e8, 26)
fig, (ax_raw, ax_phys) = plt.subplots(1, 2, figsize=(12, 4.5))
ax_raw.hist(E_true[is_neutrino], bins=true_bins, histtype='step', lw=2, color='black')
ax_raw.set(xscale='log', yscale='log', xlabel=r'True neutrino energy [GeV]',
           ylabel='Number of selected MC entries (not events expected)',
           title='Artificial sampling of selected MC')
for gamma in indices:
    wg = astro_counts(gamma)
    ax_phys.hist(E_true[is_neutrino], bins=true_bins, weights=wg[is_neutrino],
                 histtype='step', lw=1.8, label=fr'$\gamma={gamma:g}$')
ax_phys.axvline(E0_GEV, color='grey', ls='--', lw=1, label='100 TeV pivot')
ax_phys.set(xscale='log', yscale='log', xlabel=r'True neutrino energy [GeV]',
            ylabel='Expected detected neutrino events / bin',
            title='Physical flux × MC normalization × time')
ax_phys.legend(fontsize=8)
fig.tight_layout()
plt.show()

# At fixed Phi(E0), for E<E0 larger gamma raises the flux; for E>E0 it lowers it.
# Which gamma wins for the total is a property of both flux AND energy-dependent acceptance.


**Instructor notes — Exercise 3.** The raw MC distribution reflects the *simulation's
 sampling strategy and selection*, not a physical exposure-weighted rate. With fixed
 $\Phi(E_0)$, a larger $\gamma$ gives a larger flux for $E<E_0$ and a smaller flux
 for $E>E_0$. Consequently, at fixed $E_0$ the prediction for the **below-pivot
 contribution** increases with $\gamma$, while the **above-pivot contribution**
 decreases. There is no universal ordering of their **sum**: effective area and the
 accepted true-energy range determine which part dominates. Changing the pivot without
 consistently changing the normalization defines a *different physical model*;
 reparameterizing the same power law does not change predictions.

### Exercise 4 — atmospheric versus astrophysical fluxes

1. Plot `pionFlux`, `kaonFlux`, `promptFlux` against MC **true neutrino energy**. Compare them with astrophysical hypotheses $\Phi_0(E/E_0)^{-\gamma}$, $\gamma=2,2.7,3.8$. State units.
2. Why is there a **spread** of atmospheric flux values at the same energy? Test a restricted band in true zenith. The HESE arrays contain **event-wise flux evaluated at each MC truth configuration**, not a direction-independent 1-D flux table.
3. Now turn flux into predicted detected events: apply the **appropriate veto passing fractions** to conventional/prompt atmospheric neutrinos and multiply by `w0 × T`. Add atmospheric muons using their *separate* weight. What happens if prompt is multiplied by 0, 1, and 3?

In [ ]:
# Exercise 4 — event-wise atmospheric fluxes versus assumed astrophysical fluxes
true_coszen = np.cos(df_mc['primaryZenith'].to_numpy(float))
zenith_slice = is_neutrino & np.isfinite(true_coszen) & (np.abs(true_coszen) < 0.25)
rng = np.random.default_rng(7)  # reproducible, avoids drawing all MC points
valid_indices = np.flatnonzero(zenith_slice)
subset = rng.choice(valid_indices, size=min(3500, len(valid_indices)), replace=False)
fig, ax = plt.subplots(figsize=(9, 5))
for field, label in [('pionFlux', 'Atmospheric ν from π'),
                     ('kaonFlux', 'Atmospheric ν from K'),
                     ('promptFlux', 'Prompt atmospheric ν')]:
    values = df_mc[field].to_numpy(float)[subset]
    good = np.isfinite(values) & (values > 0)
    ax.scatter(E_true[subset][good], values[good], s=7, alpha=.22, label=label)
energy_grid = np.geomspace(1e2, 1e8, 250)
for gamma in [2.0, 2.7, 3.8]:
    ax.plot(energy_grid, astro_flux(energy_grid, gamma), lw=1.9,
            label=fr'Astrophysical ν, $\gamma={gamma:g}$')
ax.set(xscale='log', yscale='log', xlabel=r'True neutrino energy [GeV]',
       ylabel=r'Differential flux [GeV$^{-1}$ s$^{-1}$ sr$^{-1}$ cm$^{-2}$]')
ax.legend(ncol=2, fontsize=8)
plt.show()

# The event-wise atmospheric fluxes are not the event counts: apply weights, livetime and veto.
for prompt_scale in [0., 1., 3.]:
    components_atm = background_counts(prompt_scale=prompt_scale, self_veto=True)
    print(f'Prompt normalization x{prompt_scale:g}:',
          {key: round(float(value.sum()), 3) for key, value in components_atm.items()})
print('With self-veto corrections:',
      {k: round(float(v.sum()), 3) for k, v in background_counts().items()})
print('Without self-veto corrections (comparison only):',
      {k: round(float(v.sum()), 3) for k, v in background_counts(self_veto=False).items()})

# Sanity check: conventional = (π + K) × conventional veto × w0 × T, neutrinos only.
manual_conventional = np.where(
    is_neutrino,
    T*w0*(df_mc['pionFlux'].to_numpy(float) + df_mc['kaonFlux'].to_numpy(float))
    *df_mc['conventionalSelfVetoCorrection'].to_numpy(float), 0.)
np.testing.assert_allclose(np.nan_to_num(manual_conventional),
                           background_counts()['Conventional atm. ν'])


**Instructor notes — Exercise 4.** `pionFlux`, `kaonFlux` and `promptFlux` are *fluxes
 evaluated at the MC event's true energy, direction and particle species*. Their spread
 at a given energy reflects the other truth coordinates; even the narrow-zenith plot
 mixes different species and zeniths. Conventional flux is the **sum** of pion and kaon
 terms and uses `conventionalSelfVetoCorrection`; prompt uses `promptSelfVetoCorrection`.
 Atmospheric muons have their own *rate* normalization and must not be reweighted
 with a neutrino flux. Varying prompt normalization changes only the prompt component
 in this simple exercise; a full analysis would also consider correlated uncertainties.

## C. Detector response — what can we really derive?

The published sample contains selected MC events *with truth and reconstructed quantities*. Thus:

- **Absolute HESE effective area** in a specified true-energy/true-direction/particle-species bin: **yes**, via `weightOverFluxOverLivetime`; no unsimulated events required because this field encodes the generation normalization. This is an *analysis-selection* effective area, not a raw geometrical or trigger area.
- **Energy and zenith response of selected HESE events:** **yes**, as conditional migration matrices and weighted bias/quantile curves.
- **Conditional probability of reconstructed morphology given neutrino flavor + interaction type among selected events:** **yes**. It is *not* a truth-versus-reconstructed morphology confusion matrix unless a truth morphology label exists.
- **True 3-D angular separation, full PSF, 50%/90% angular containment:** **no**. The public JSON has true/reco zenith but **not both azimuths**, so `abs(recoZenith-primaryZenith)` is **not** an angular error.
- **Trigger efficiency, pre-selection-to-final selection efficiency:** **no**, because the release does not contain a matched complete unselected/generated event sample.  
The toy notebook will cover full angular response/PSF, and a directly measurable selection efficiency.

**Suggested pacing:** prioritize Exercises 1–3 (weights and data/MC), Exercise 4 (flux components), and the 1D effective area plus energy migration in Exercises 5–6. The 2D effective area, bias quantiles, and Exercise 7 are extensions for students who finish early.


### Exercise 5 — derive the effective area from public HESE weights

Following lecture slides 56–61, for a bin $\Delta E\,\Delta\Omega$, derive:
$$
A_{\rm eff}(E,\Omega)\simeq
\frac{\sum_{i\in\text{selected bin}}\mathrm{weightOverFluxOverLivetime}_i}
{\Delta E\,\Delta\Omega}.
$$

1. Restrict to **neutrinos of one PDG species** (e.g. `primaryType == 14` for $\nu_\mu$, not the sum over all flavors), then bin in **true energy**. Calculate the sky-averaged $A_{\rm eff}(E)$ by dividing by $4\pi\,\Delta E$. Label the result in **cm²** (optionally convert to m² by dividing by 10⁴).
2. **Optional:** build a 2D $A_{\rm eff}(E,\cos\theta)$: $\Delta\Omega = 2\pi\,\Delta\cos\theta$. Plot it as a heatmap; identify energy/zenith dependence.
3. What changes for a different neutrino species? Does changing $\gamma$ change the effective area itself?

**Hint:** this public field is *not* an astrophysical event weight and contains no dependence on your chosen flux. Do **not** divide by the generated event count again.

In [ ]:
# Exercise 5 — absolute HESE *selected-sample* effective area
pdg = df_mc['primaryType'].to_numpy(int)
e_edges = np.geomspace(1e2, 1e8, 22)
fig, ax = plt.subplots()
for species, label in [(14, r'$\nu_\mu$'), (-14, r'$\bar{\nu}_\mu$')]:
    sel = is_neutrino & (pdg == species)
    sum_w0, _ = np.histogram(E_true[sel], bins=e_edges, weights=w0[sel])
    ae_cm2 = sum_w0 / (np.diff(e_edges) * (4*np.pi))
    ae_m2 = ae_cm2/1e4
    # A zero bin has no log-scale value (do not infer zero efficiency from an empty MC bin).
    ax.stairs(np.where(ae_m2 > 0, ae_m2, np.nan), e_edges, label=label)
    print(f'{label}: {sel.sum()} selected MC entries')
ax.set(xscale='log', yscale='log', xlabel=r'True neutrino energy [GeV]',
       ylabel=r'Sky-averaged selected HESE $A_{\rm eff}$ [m$^2$]')
ax.legend(); plt.show()

# Optional: the directional effective area Aeff(E, cos zenith) for νμ.
nu_mu = is_neutrino & (pdg == 14)
cz_true = np.cos(df_mc['primaryZenith'].to_numpy(float))
cz_edges = np.linspace(-1., 1., 9)
sum_w0_2d, _, _ = np.histogram2d(
    E_true[nu_mu], cz_true[nu_mu], bins=[e_edges, cz_edges], weights=w0[nu_mu])
delta_E = np.diff(e_edges)[:, None]
delta_omega = (2*np.pi*np.diff(cz_edges))[None, :]
ae_2d_cm2 = sum_w0_2d/(delta_E*delta_omega)
fig, ax = plt.subplots()
heat = np.ma.masked_less_equal((ae_2d_cm2/1e4).T, 0)
if heat.count() > 0:
    mesh = ax.pcolormesh(e_edges, cz_edges, heat, norm=LogNorm(), shading='auto')
    fig.colorbar(mesh, ax=ax, label=r'Selected HESE $A_{\rm eff}$ [m$^2$]')
ax.set(xscale='log', xlabel=r'True neutrino energy [GeV]',
       ylabel=r'True $\cos\theta_z$', title=r'$\nu_\mu$ direction-dependent effective area')
plt.show()

# Angularly integrating Aeff(E,cos θ) over dΩ/4π should reproduce the sky average.
nu_mu_w_sum, _ = np.histogram(E_true[nu_mu], bins=e_edges, weights=w0[nu_mu])
ae_1d_mu_cm2 = nu_mu_w_sum/(np.diff(e_edges)*4*np.pi)
ae_1d_from_2d_cm2 = (ae_2d_cm2*np.diff(cz_edges)[None, :]).sum(axis=1)/2.
np.testing.assert_allclose(ae_1d_mu_cm2, ae_1d_from_2d_cm2, rtol=1e-9, atol=1e-9)


**Instructor notes — Exercise 5.** For **one PDG species**, the quantity
 $\sum w_0/(\Delta E\,\Delta\Omega)$ is an effective area for the released HESE
 *analysis selection*. For an all-sky-average in an energy bin, use
 $\Delta\Omega=4\pi$; for a $\cos\theta_z$ bin, use
 $\Delta\Omega=2\pi\,\Delta\cos\theta_z$. Do not multiply by an astrophysical flux
 or divide again by the number of generated MC events. Comparing $\nu_\mu$ with
 $\bar\nu_\mu$ is meaningful because they are **distinct species**. The response
 is not a function of the chosen power-law index $\gamma$; any visible index
 dependence would arise from how the response was binned, averaged or weighted.

### Exercise 6 — reconstructed energy response and bias

1. Use `primaryEnergy` vs `recoDepositedEnergy` for MC neutrinos. Construct a **weighted 2D histogram**, then normalize each **true-energy** row to get $P(E_{\rm dep,reco}\ \text{bin}\mid E_{\nu,\rm true}\ \text{bin}, \mathrm{selected})$.
2. **Optional extension:** compute $\delta=\log_{10}(E_{\rm dep,reco}/E_{\nu,\rm true})$. In true-energy bins plot the **weighted median** and the **central 68% interval** (quantiles 16%, 50%, 84%). Try $\gamma=2$ and $3.8$.
3. Why does this response contain interaction/invisible-energy effects *as well as* reconstruction error? Why may weighted quantiles change when combining wide energy bins or topologies?

4. **Discussion:** If you construct an energy migration matrix using different astrophysical spectral indices, should the matrix change? Under which conditions would you expect it to be independent of the assumed spectrum? Think about finite true-energy bins, energy–topology correlations, and how the simulated events are weighted.


In [ ]:
# Exercise 6 — deposited-energy migration for *selected* neutrino events
E_dep = df_mc['recoDepositedEnergy'].to_numpy(float)
good = is_neutrino & np.isfinite(E_dep) & (E_dep > 0)
true_edges = np.geomspace(1e2, 1e8, 16)
reco_edges = np.geomspace(1e1, 1e9, 22)

# A row is a finite true-energy bin, a column a reconstructed deposited-energy bin.
weight_gamma27 = astro_counts(2.7)
H, _, _ = np.histogram2d(E_true[good], E_dep[good],
                         bins=(true_edges, reco_edges), weights=weight_gamma27[good])
row_sum = H.sum(axis=1, keepdims=True)
P = np.divide(H, row_sum, out=np.zeros_like(H), where=row_sum > 0)
fig, ax = plt.subplots()
mesh = ax.pcolormesh(true_edges, reco_edges, P.T, shading='auto')
fig.colorbar(mesh, ax=ax,
             label='P(deposited-energy bin | true-energy bin, selected)')
ax.set(xscale='log', yscale='log', xlabel=r'True neutrino energy [GeV]',
       ylabel=r'Reconstructed deposited energy [GeV]')
plt.show()
print('Non-empty true-energy rows normalize to 1:',
      np.allclose(P[row_sum[:, 0] > 0].sum(axis=1), 1))
print('Empty true-energy bins (undefined response):', np.count_nonzero(row_sum[:, 0] == 0))

# Optional: median and central 68% range in bins of *true* energy.
delta = np.log10(E_dep[good]/E_true[good])  # log10(Edep,reco / Eν,true)
centers = np.sqrt(true_edges[:-1]*true_edges[1:])
fig, ax = plt.subplots()
for gamma in [2.0, 3.8]:
    event_weights = astro_counts(gamma)[good]
    quantile_bins = []
    for lo, hi in zip(true_edges[:-1], true_edges[1:]):
        in_bin = (E_true[good] >= lo) & (E_true[good] < hi)
        quantile_bins.append(weighted_quantile(delta[in_bin], event_weights[in_bin],
                                               [.16, .50, .84]))
    q = np.asarray(quantile_bins)
    ax.plot(centers, q[:, 1], marker='o', label=fr'Median, $\gamma={gamma:g}$')
    ax.fill_between(centers, q[:, 0], q[:, 2], alpha=.15)
ax.axhline(0, ls='--', color='grey', lw=1)
ax.set(xscale='log', xlabel='True neutrino energy [GeV]',
       ylabel=r'$\log_{10}(E_{\rm dep,reco}/E_{\nu,\rm true})$')
ax.legend(); plt.show()

# Spectral-index effect on a conditional matrix after mixing inside broad finite bins:
H38, _, _ = np.histogram2d(E_true[good], E_dep[good],
                           bins=(true_edges, reco_edges), weights=astro_counts(3.8)[good])
row38 = H38.sum(axis=1, keepdims=True)
P38 = np.divide(H38, row38, out=np.zeros_like(H38), where=row38 > 0)
valid_rows = (row_sum[:, 0] > 0) & (row38[:, 0] > 0)
print('Max absolute matrix element change γ=2.7 -> 3.8 in nonempty rows:',
      np.max(np.abs(P[valid_rows] - P38[valid_rows])) if np.any(valid_rows) else np.nan)


**Instructor notes — Exercise 6.** The matrix rows represent **finite bins in true
 neutrino energy** and are normalized across reconstructed **deposited-energy** bins.
 It is $P(E_{\rm dep,reco}\text{ bin}\mid E_{\nu,\rm true}\text{ bin},\,\mathrm{selected})$,
 **not** a detection probability before selection. Deposited energy is different from
 incoming neutrino energy because the interaction type, outgoing invisible particles,
 and reconstruction all matter. When the source spectrum changes, the conditional
 matrix can change in *finite bins* through different within-bin energy/zenith/flavor
 mixtures; for a sufficiently fine fully conditioned detector response (with correct
 statistical support), the intrinsic detector kernel should not depend on the assumed
 incident flux. Empty rows have no measured response, not a true probability of zero.

### Exercise 7 — zenith migration and conditional morphology (optional extension)

1. Make a weighted migration matrix from $\cos(\theta_{\rm true})$ to $\cos(\theta_{\rm reco})$, normalized by true-zenith bin.
2. For selected neutrino MC, compute $P(\mathrm{recoMorphology}\mid\mathrm{interactionType},\mathrm{selected})$ (or condition also on true flavor). A conditional **morphology response** can be made, but not a true topology efficiency because the public release has no `trueMorphology` label.
3. Can you plot the full 3-D angular resolution or PSF? Which two variables are missing? Do **not** interpret a simple zenith difference as a 3D opening angle.

In [ ]:
# Exercise 7 — zenith migration and morphology, both conditional on selected HESE MC
cz_true = np.cos(df_mc['primaryZenith'].to_numpy(float))
cz_reco = np.cos(df_mc['recoZenith'].to_numpy(float))
valid = is_neutrino & np.isfinite(cz_true) & np.isfinite(cz_reco)
ct_edges = np.linspace(-1., 1., 13)
H, _, _ = np.histogram2d(cz_true[valid], cz_reco[valid],
                         bins=(ct_edges, ct_edges), weights=astro_counts(2.7)[valid])
row = H.sum(axis=1, keepdims=True)
P_zenith = np.divide(H, row, out=np.zeros_like(H), where=row > 0)
fig, ax = plt.subplots()
m = ax.pcolormesh(ct_edges, ct_edges, P_zenith.T, shading='auto')
fig.colorbar(m, ax=ax, label='P(reconstructed bin | true bin, selected)')
ax.set(xlabel=r'True $\cos\theta_z$', ylabel=r'Reconstructed $\cos\theta_z$')
plt.show()

interaction = df_mc['interactionType'].to_numpy(float)
morph = df_mc['recoMorphology'].to_numpy(float)
valid = is_neutrino & np.isfinite(interaction) & np.isfinite(morph)
H_morph, _, _ = np.histogram2d(interaction[valid], morph[valid],
                              bins=([.5, 1.5, 2.5, 3.5], [-.5, .5, 1.5, 2.5]),
                              weights=astro_counts(2.7)[valid])
row_morph = H_morph.sum(axis=1, keepdims=True)
Q = np.divide(H_morph, row_morph, out=np.full_like(H_morph, np.nan), where=row_morph > 0)
print('Rows: true interaction CC, NC, GR; columns: reconstructed cascade, track, double cascade')
display(pd.DataFrame(Q, index=['CC', 'NC', 'GR'],
                     columns=['cascade', 'track', 'double cascade']).round(3))

# Extension: the reconstructed morphology depends strongly on *both* flavor and interaction.
flavor = np.abs(df_mc['primaryType'].to_numpy(int))
for pdg_flavor, name in [(12, 'electron'), (14, 'muon'), (16, 'tau')]:
    one_flavor_cc = valid & (interaction == 1) & (flavor == pdg_flavor)
    counts = np.bincount(morph[one_flavor_cc].astype(int),
                         weights=astro_counts(2.7)[one_flavor_cc], minlength=3)[:3]
    if counts.sum() > 0:
        print(f'CC {name} (ν + anti-ν), conditional morphology [cascade, track, double]:',
              np.round(counts/counts.sum(), 3))


**Instructor notes — Exercise 7.** Zenith migration is only a **one-coordinate
 angular response**. A full 3-D opening angle $\Delta\psi$ requires **both** the true
 and reconstructed azimuth as well as the two zeniths; these azimuth fields are absent
 from the supplied HESE JSON. The table estimates
 $P(\mathrm{recoMorphology}\mid\mathrm{interactionType},\mathrm{selected})$,
 optionally conditioned on flavor. It is **not** a truth-morphology confusion matrix
 or pre-selection topology efficiency. Rows with no weighted entries are undefined.

## D. Reflection and extensions

1. How would you use $T\int dE\,d\Omega\, A_{\rm eff}(E,\Omega)\,\Phi(E,\Omega)$ to predict the number of events for a flux hypothesis? What is the meaning of each integration measure?
2. Why might a steeper spectrum produce *more* observed events under one threshold/pivot and *fewer* under another?
3. Which detector-response properties are intrinsic to the selected simulation, and which plots depend on the assumed flux **because of population mixing**?
4. In a full analysis, how would you propagate finite MC statistics and detector systematics? Why can't these plots alone establish the best-fit astrophysical spectrum?

**Optional advanced exercise:** compute the *effective sample size* $N_{\rm eff}=(\sum_i w_i)^2/\sum_i w_i^2$ in true-energy bins. Identify where simulation statistics are thin.

**Further reading:** [IceCube release README](https://github.com/icecube/HESE-7-year-data-release) and released `plot_effective_areas.py`. The other notebook constructs a toy where the generated sample and full true/reconstructed directions *are known*, permitting direct selection-efficiency and PSF exercises.

### Instructor discussion — Reflection and extensions

1. For each neutrino species, $N_{\rm exp}=T\int dE\,d\Omega\,A_{\rm eff}(E,\Omega)\Phi(E,\Omega)$;
   $dE$ is the true-energy measure, $d\Omega$ is the true-direction solid-angle
   measure (for an azimuth-symmetric detector, $d\Omega=2\pi\,d\cos\theta$). The
   effective area includes the interaction, detector and **analysis selection** response.
2. At **fixed flux at the pivot**, raising $\gamma$ raises the low-energy flux and
   decreases the high-energy flux. The relative count contributions are controlled
   by the energy-dependent effective area and thresholds, so the total count ordering
   is not universal.
3. The underlying response at fixed **complete true event configuration** is detector
   dependent, not flux dependent. Flux-weighted **finite-bin averages** of the energy
   migration matrix, the morphology mixture, and angular projections may change with
   spectral index due to different mixtures of events inside bins.
4. For the full physics inference, finite weighted MC statistics require treatment
   beyond simple histograms (for example effective-sample-size diagnostics or suitable
   likelihood methods). Detector systematics, flux normalization and shape, the
   atmospheric components, event selection and Poisson likelihood all matter.
   **None of these plots fits a spectral index to the data.**

**Optional advanced solution:** evaluate the effective MC sample size
$N_{\rm eff}=(\sum_i w_i)^2/\sum_i w_i^2$ per true-energy bin (only as a diagnostic,
not a substitute for a full finite-MC uncertainty treatment).

In [ ]:
# Optional advanced exercise — effective sample size, using expected astro weights
edges = np.geomspace(1e2, 1e8, 20)
w = astro_counts(2.7)
sel = is_neutrino & np.isfinite(w) & (w > 0)
sumw, _ = np.histogram(E_true[sel], bins=edges, weights=w[sel])
sumw2, _ = np.histogram(E_true[sel], bins=edges, weights=w[sel]**2)
neff = np.divide(sumw**2, sumw2, out=np.full_like(sumw, np.nan), where=sumw2>0)
fig, ax = plt.subplots()
ax.stairs(neff, edges)
ax.set(xscale='log', yscale='log', xlabel=r'True $E_\nu$ [GeV]',
       ylabel=r'Weighted $N_{\rm eff}$',
       title='Where are weighted MC statistics limited?')
plt.show()
print('Minimum nonempty-bin effective sample size:',
      np.nanmin(neff) if np.any(np.isfinite(neff)) else 'none')